In [1026]:
import numpy as np


In [1027]:
def generate_col_row_ls (seq,gap):
    row_or_col_ls = []
    n = 0 
    value = 0
    while n < (len(seq) + 1):
        row_or_col_ls.append(value)
        value += gap
        n += 1

    return row_or_col_ls


In [1028]:
def initiate_needleman_matrix(seq1,seq2,gap):
    row_ls=generate_col_row_ls(seq1,gap)
    col_ls =generate_col_row_ls(seq2,gap)
    mat = np.zeros((len(col_ls), len(row_ls)), dtype=int)
    mat[0,:] = row_ls
    mat[:,0] = col_ls
    return mat




In [1029]:

def across(mat, gap, i, j):
    across = mat[i, j-1] + gap
    return across

def vertical (mat, gap, i, j):
    vertical = mat [i-1,j] + gap
    return vertical


In [1030]:
def diagonal (mat, i, j, match, miss, seq1, seq2):
    seq1_ls= ["-"]
    seq2_ls= ["-"]
    for nuc in seq1:
        seq1_ls.append(nuc)
    for nuc in seq2:
        seq2_ls.append(nuc)

    if seq2_ls[i] == seq1_ls[j]:
        diagonal = mat[i-1,j-1] + match
    else: 
        diagonal = mat[i-1,j-1] + miss
    
    return diagonal
    
 

In [1031]:
def fill_wunsch_matrix(mat,gap,match,mismatch,seq1,seq2):
    
    pointer = np.zeros((len(mat), len(mat[0])), dtype=object)

    for i in range(1, len(mat)):
        pointer[i,0] = "V"  # first column is all vertical
    for j in range(1, len(mat[0])):
        pointer[0,j] = "A"  # first row is all across

        
    for i in range(1, len(mat)):
        for j in range(1,len(mat[0])):            
            across_val = across(mat,gap,i,j)
            vertical_val = vertical(mat,gap,i,j)
            diagonal_val = diagonal(mat,i,j,match,mismatch,seq1,seq2)
            mat[i,j] = max(across_val, vertical_val, diagonal_val)

            #If diagonal is max; a character from each sequence was aligned
            if mat[i,j] == diagonal_val:
                pointer[i,j] = "D"
            #If vertical was max; a gap was introduced in sequence 1
            elif mat[i,j] == vertical_val:
                pointer[i,j] = "V"
            #if across was max; a gap was introduced in sequence 2 
            else:
                pointer[i,j] = "A"
            
    return mat, pointer



In [1032]:
def traceback(mat, seq1, seq2, pointer):
    seq1_ls = ["-"]
    seq2_ls = ["-"]

    #for each nucleotide in sequence 1; add to sequence 1 list
    for nuc in seq1:          
        seq1_ls.append(nuc)

    #for each nucleotide in sequence 2; add to sequence 1 list
    for nuc in seq2:          
        seq2_ls.append(nuc)

    #Make sequence alignment strings   
    seq1_alignment = ""
    seq2_alignment = ""

    i = len(mat) - 1
    j = len(mat[0]) - 1

    while i > 0 or j > 0: #while the matrix is not at the top first row and first column 
        if pointer[i,j] == "D": #if the pointer is D; came diagonally
            seq1_alignment = seq1_ls[j] + seq1_alignment #the current nucleotide gets added to the list
            seq2_alignment = seq2_ls[i] + seq2_alignment #the current nucleotide gets added to the list
            
            #Move diagonally across 
            i -= 1 
            j -= 1

        elif pointer[i,j] == "V": #If the pointer is equal to vertical; came from above
            seq1_alignment = "-" + seq1_alignment #Add gap to the aligned one list 
            seq2_alignment = seq2_ls[i] + seq2_alignment #add the current nucleotide from sequence 2
            i -= 1 #move up one row

        else:
            seq1_alignment = seq1_ls[j] + seq1_alignment #add current nucleotide row from seq
            seq2_alignment = "-" + seq2_alignment #add gap into lost for sequence twos alignment 
            j -= 1 #move left one column
    
    return seq1_alignment, seq2_alignment

In [1033]:
def needleman_wunsch_alignment(seq1,seq2,gap,match,mismatch):
    mat =initiate_needleman_matrix(seq1,seq2,gap)
    mat, pointer = fill_wunsch_matrix(mat,gap,match,mismatch,seq1,seq2)
    seq1_alignment, seq2_alignment = traceback(mat, seq1, seq2, pointer)
    print(f"Sequence 1: {seq1_alignment}")
    print(f"Sequence 2: {seq2_alignment}")
    return seq1_alignment, seq2_alignment


In [1034]:
sequence1 = "ATCATCGGC" #seq2
sequence2 = "AGGGATCGGC" #seq1

In [1035]:
match = 1 #points if it is a mathc
mismatch = -1 #points if it is not a match 
gap = -2 # points if it is a gap

In [1036]:
needleman_wunsch_alignment(sequence1,sequence2, gap, match, mismatch)

Sequence 1: A-TCATCGGC
Sequence 2: AGGGATCGGC


('A-TCATCGGC', 'AGGGATCGGC')